# HinGE - Difficulty Assignment (English to Hinglish generation)

Assigns `difficulty` (Easy / Medium / Hard) to 200 sampled HinGE rows.

**Task.** `question` is an English sentence, `answer` is a romanised
Hindi-English (Hinglish) rendering of it. The models must generate Hinglish -
no options. Note this is the **opposite direction** to PHINC.

**Two-pass design.** A generation score is continuous, so:

1. **Score** every row with every model and store the raw chrF++ value.
2. **Derive a threshold** (Cell 9); a model passes a row when it clears the
   threshold, and the three votes sum as usual:

| Models passing | Difficulty |
|---|---|
| 3 / 3 | Easy |
| 2 / 3 | Medium |
| 0-1 / 3 | Hard |

Raw scores are stored, so re-thresholding costs nothing - no model is re-run.

**The measurement problem, stated up front.** Hinglish keeps much of the
English input verbatim - measured on this file, **46.6% of target words are
copied straight from the source**. So a system that outputs the English
unchanged already scores about **56% chrF++ / 47% ROUGE-L**. The usable range
is roughly 56%-100%, far narrower than a normal translation task, and any
threshold must sit clearly above that floor. Cell 4 measures it, and the
default `THRESHOLD_MODE` is `"floor_margin"` for exactly this reason: it pins
the pass mark a fixed margin *above* the do-nothing floor rather than trusting
a median that could sit only a point or two above it.

**Multiple references are used where they exist.** Two source sentences appear
twice with different valid Hinglish renderings. Those are genuine alternative
references, not duplicates, so Cell 4 groups them and scores against **all** of
them - chrF++ takes the best match. Valid Hinglish varies enormously in
spelling and in how much gets translated, so scoring against a single reference
understates every model.

**Output.** `hinge_difficulty.jsonl` - the original 14 schema fields with
`difficulty` filled in, plus an audit file with every model's output.

### Cell 1 - Install dependencies and authenticate

Adds `sacrebleu` for the reference chrF++ implementation.

**An HF token is required.** Llama-3.1 and Gemma-2 are gated; only Mistral is
open. Accept each licence on huggingface.co, create a **read** token, then add
it in Colab via the **key icon** as a secret named `HF_TOKEN`. Use the secret
rather than pasting the token into a cell.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!ls -lah /content/drive/
!ls -lah /content/drive/MyDrive/

total 16K
dr-x------ 4 root root 4.0K Sep  3 03:30 .Encrypted
drwx------ 3 root root 4.0K Sep  3 03:30 MyDrive
dr-x------ 3 root root 4.0K Sep  3 03:30 .shortcut-targets-by-id
drwx------ 5 root root 4.0K Sep  3 03:30 .Trash-0
total 8.6M
-rw------- 1 root root 110K Jul 27  2024 'AdmitCard (1).jpg'
-rw------- 1 root root 110K Jul 27  2024  AdmitCard.jpg
drwx------ 2 root root 4.0K Oct 13  2025 'Colab Notebooks'
-rw------- 1 root root 265K Jul 27  2024  download.pdf
-rw------- 1 root root  38K Jun 22  2023 'JEE ADV.jpg'
-rw------- 1 root root 265K Apr 29  2023 'jee main result.pdf'
-rw------- 1 root root 5.4M Jan 28  2023 'jee mains admit card.pdf'
-rw------- 1 root root  172 Jul 30  2024 'Jobs Profile.gdoc'
lrw------- 1 root root    0 Sep  3 03:30  models -> /content/drive/.shortcut-targets-by-id/1ol4XKxMrgF8s3NuL2njvzfjSR6xU2hXw/models
-rw------- 1 root root 2.3M Aug 20  2024 'orassignment2 .pdf'
-rw------- 1 root root  45K Jul 27  2024  passport.jpg
-rw------- 1 root root  86K Jan 20  

In [5]:
!ls -lah /content/drive/MyDrive/models/

total 12K
drwx------ 2 root root 4.0K Sep  2 20:14 gemma_4bit
drwx------ 2 root root 4.0K Sep  2 19:14 llama_4bit
drwx------ 2 root root 4.0K Sep  2 18:45 mistral_4bit


In [6]:
!pip -q install -U transformers accelerate bitsandbytes huggingface_hub sacrebleu

import sacrebleu
print("sacrebleu", sacrebleu.__version__)

HF_OK = False
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"))
    HF_OK = True
    print("HF login OK")
except Exception as e:
    print("No HF token ({}: {})".format(type(e).__name__, e))
    print("Mistral will still work; gated Llama/Gemma will fail with a 401.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 11.9 MB/s eta 0:00:00
sacrebleu 2.6.0
HF login OK


### Cell 2 - Mount Drive

Drive is the weight cache: models are downloaded once, quantised to 4-bit and
saved here, so later runs skip the download.

`DRIVE_OK` records whether the mount actually succeeded. If you see
**"credential propagation was unsuccessful"**, the auth popup did not complete:
re-run and finish it, allow pop-ups and third-party cookies for
`colab.research.google.com`, or mount from the Files sidebar.

In [7]:
import os

DRIVE_OK    = False
DRIVE_MOUNT = "/drive"
CACHE_DIR   = os.path.join(DRIVE_MOUNT, "MyDrive", "models")

try:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT, force_remount=True)
    DRIVE_OK = os.path.isdir(os.path.join(DRIVE_MOUNT, "MyDrive"))
except ImportError:
    print("Not running on Colab - Drive caching disabled.")
except Exception as e:
    print("DRIVE MOUNT FAILED: {}".format(e))

if DRIVE_OK:
    os.makedirs(CACHE_DIR, exist_ok=True)
    print("Drive mounted | weight cache: {}".format(CACHE_DIR))
else:
    print("\nWARNING: no Drive - weights will NOT be cached between sessions.")

Mounted at /drive
Drive mounted | weight cache: /drive/MyDrive/models


### Cell 3 - Configuration

- `THRESHOLD_MODE` - how Cell 9 picks the pass mark.
  - `"floor_margin"` (**default here**) - the copy-the-input floor plus
    `FLOOR_MARGIN`. The right choice for this dataset: the floor is around 56%,
    so a model must beat "emit the English unchanged" by a real margin to pass.
  - `"auto_median"` - median of pooled model scores. Gives a balanced split,
    but on HinGE it can land only a point or two above the floor, which would
    pass models that barely translated anything. Cell 9 warns if it does.
  - `"fixed"` - use `FIXED_THRESHOLD` verbatim.
- `SET_EVAL_METRIC` - leave `None` to keep the file's existing `eval_metric`.
  Set to `"chrF++"` to record the metric actually used here.
- `MODELS` - three judges from three families (Mistral / Meta / Google) so
  their errors decorrelate. `USE_INSTRUCT` picks instruction-tuned checkpoints;
  Cell 5 detects base vs chat automatically either way.

In [8]:
import gc
import re
import json
import random
import shutil
import statistics
from collections import Counter, defaultdict

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ---- paths ----
INPUT_FILE  = "hinge.jsonl"
OUTPUT_FILE = "hinge_difficulty.jsonl"
AUDIT_FILE  = "hinge_audit.jsonl"
PROG_DIR    = "judge_progress"

# ---- sampling ----
N_ROWS        = 200
SEED          = 42
MIN_REF_WORDS = 3

# ---- thresholding ----
THRESHOLD_MODE  = "floor_margin"    # floor_margin | auto_median | fixed
FLOOR_MARGIN    = 0.10              # how far above the do-nothing floor
FIXED_THRESHOLD = 0.70

# ---- generation ----
MAX_NEW_TOKENS = 320
BATCH_SIZE     = 25

# ---- schema ----
SET_EVAL_METRIC = None

# ---- the three judges ----
USE_INSTRUCT = True

REPOS = {
    True: {
        "mistral": "mistralai/Mistral-7B-Instruct-v0.3",   # ungated
        "llama":   "meta-llama/Llama-3.1-8B-Instruct",     # GATED
        "gemma":   "google/gemma-2-9b-it",                 # GATED
    },
    False: {
        "mistral": "mistralai/Mistral-7B-v0.3",
        "llama":   "meta-llama/Llama-3.1-8B",
        "gemma":   "google/gemma-2-9b",
    },
}[USE_INSTRUCT]

MODELS = [
    {"name": "mistral", "repo": REPOS["mistral"]},
    {"name": "llama",   "repo": REPOS["llama"]},
    {"name": "gemma",   "repo": REPOS["gemma"], "attn": "eager"},
]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,     # T4 has no bf16
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

SCHEMA_KEYS = [
    "id", "source", "category", "subcategory", "region", "language",
    "difficulty", "task_type", "question", "options", "answer",
    "explanation", "cultural_attr", "eval_metric",
]

os.makedirs(PROG_DIR, exist_ok=True)
print("Judges ({}):".format("instruct" if USE_INSTRUCT else "base"))
for m in MODELS:
    cached = DRIVE_OK and os.path.isfile(
        os.path.join(CACHE_DIR, m["name"] + "_4bit", "config.json"))
    print("  {:<8} {:<42} {}".format(
        m["name"], m["repo"], "cached in Drive" if cached else "will download"))
print("\ndevice:", "cuda" if torch.cuda.is_available() else "CPU (will be very slow)")

Judges (instruct):
  mistral  mistralai/Mistral-7B-Instruct-v0.3         cached in Drive
  llama    meta-llama/Llama-3.1-8B-Instruct           cached in Drive
  gemma    google/gemma-2-9b-it                       cached in Drive

device: cuda


### Cell 4 - Load, group references, sample, and measure the floor

Three things happen here.

**Multiple references are grouped.** Two English sentences appear twice with
different valid Hinglish renderings. `REFS` maps each source to every reference
it has, and scoring uses them all - chrF++ takes the best match. Hinglish is
not a single correct string, so scoring against one arbitrary rendering
penalises valid output.

**The sample is drawn under a fixed seed**, so the same rows return on every
run - which is what makes the resume logic safe across sessions.

**The floors are measured.** Two model-free reference points:

- **Copy the English input unchanged** - the score for doing nothing. On this
  dataset it is high (~56% chrF++) because Hinglish legitimately keeps much of
  the English. This is the number that matters most; Cell 9 anchors the
  threshold to it.
- **An unrelated row's Hinglish** - the true wrong-answer level.

In [11]:
import sacrebleu
_CHRF = sacrebleu.CHRF(word_order=2)          # word_order=2 makes this chrF++


def chrf_pp(hypothesis, references):
    # references is a LIST - chrF++ scores against the best match
    if not hypothesis or not hypothesis.strip():
        return 0.0
    return _CHRF.sentence_score(hypothesis, list(references)).score / 100.0


with open(INPUT_FILE, encoding="utf-8") as f:
    all_rows = [json.loads(line) for line in f]
print("Loaded {} rows from {}".format(len(all_rows), INPUT_FILE))


def usable(row):
    ref = str(row.get("answer") or "").strip()
    return len(ref.split()) >= MIN_REF_WORDS and bool(row.get("question", "").strip())


usable_rows = [r for r in all_rows if usable(r)]
if len(usable_rows) < len(all_rows):
    print("dropped {} rows with unusable references".format(
        len(all_rows) - len(usable_rows)))

# ---- group alternative references by source sentence ----
REFS = defaultdict(list)
for r in usable_rows:
    REFS[r["question"]].append(str(r["answer"]))
multi = {q: v for q, v in REFS.items() if len(v) > 1}
print("sources with more than one reference: {} ({} rows)".format(
    len(multi), sum(len(v) for v in multi.values())))

# one entry per SOURCE, so a multi-reference sentence is not sampled twice
seen, pool = set(), []
for r in usable_rows:
    if r["question"] in seen:
        continue
    seen.add(r["question"])
    pool.append(r)
print("distinct sources available: {}".format(len(pool)))
assert len(pool) >= N_ROWS, "not enough rows to sample from"

random.seed(SEED)
sample = random.sample(pool, N_ROWS)

lens = sorted(len(r["question"].split()) for r in sample)
print("\nSampled {} rows | source words: min {}, median {}, max {}".format(
    len(sample), lens[0], lens[len(lens) // 2], lens[-1]))

# ---- model-free floors ----
copy_scores = [chrf_pp(r["question"], REFS[r["question"]]) for r in sample]
rand_scores = [chrf_pp(str(sample[(i + 7) % len(sample)]["answer"]),
                       REFS[r["question"]])
               for i, r in enumerate(sample)]

COPY_FLOOR = statistics.mean(copy_scores)
RAND_FLOOR = statistics.mean(rand_scores)

print("\nReference floors (chrF++, no model involved):")
print("  copy the English input unchanged : {:.1%}   <- a system that does nothing".format(COPY_FLOOR))
print("  an unrelated row's Hinglish      : {:.1%}   <- true wrong-answer level".format(RAND_FLOOR))
print("  a perfect rendering              : 100.0%")
print("\nUsable range is only {:.0%} to 100% - narrow, because Hinglish keeps".format(COPY_FLOOR))
print("much of the English verbatim. Keep the threshold well above the floor.")

print("\n--- example row ---")
print("  english :", " ".join(sample[0]["question"].split())[:96])
print("  hinglish:", " ".join(str(sample[0]["answer"]).split())[:96])

Loaded 1973 rows from hinge.jsonl
sources with more than one reference: 2 (4 rows)
distinct sources available: 1971

Sampled 200 rows | source words: min 4, median 14, max 117

Reference floors (chrF++, no model involved):
  copy the English input unchanged : 48.4%   <- a system that does nothing
  an unrelated row's Hinglish      : 11.3%   <- true wrong-answer level
  a perfect rendering              : 100.0%

Usable range is only 48% to 100% - narrow, because Hinglish keeps
much of the English verbatim. Keep the threshold well above the floor.

--- example row ---
  english : And we 're going to expand it into two simpler expressions
  hinglish: And we 're going to expand do saral bhav me.


### Cell 5 - Build the generation prompt

Few-shot examples come from **outside** the 200-row sample, so no scored row
ever has its reference shown to the model. They are picked deterministically
from `SEED`, so every model and every re-run sees the same examples.

The examples matter more than usual here. "Write this in Hinglish" is
ambiguous - how much should be translated? which words stay English? - and the
few-shot pairs are what communicate the register HinGE's annotators actually
used. Without them a model tends to either translate everything into Devanagari
or leave the English untouched.

Two builders: `build_completion` (flat text ending on a dangling `Hinglish:`,
for base checkpoints) and `build_chat_messages` (chat turns, for instruct).

In [12]:
INSTRUCTIONS = (
    "Rewrite English sentences as Hinglish - Hindi-English code-mixed text "
    "written in the Roman alphabet.\n\n"
    "Translate some words and phrases into Hindi but write them in Roman "
    "letters, not Devanagari. Keep English words that Hindi speakers normally "
    "keep in English - technical terms, proper nouns, and common loanwords. "
    "The result should read the way an Indian speaker would naturally mix the "
    "two languages.\n\n"
    "Output only the Hinglish sentence - no explanation, no Devanagari, no "
    "repetition of the English."
)


def flat(text):
    return " ".join(str(text).split())


def pick_fewshot(k=3):
    used = {r["question"] for r in sample}
    cand = [r for r in usable_rows
            if r["question"] not in used
            and 8 <= len(r["question"].split()) <= 24]
    random.Random(SEED + 1).shuffle(cand)
    return cand[:k]


FEWSHOT = pick_fewshot()


def build_completion(source):
    text = INSTRUCTIONS + "\n"
    for r in FEWSHOT:
        text += "\nEnglish: {}\nHinglish: {}\n".format(
            flat(r["question"]), flat(r["answer"]))
    text += "\nEnglish: {}\nHinglish:".format(flat(source))
    return text


def build_chat_messages(source):
    msgs = [{"role": "system", "content": INSTRUCTIONS}]
    for r in FEWSHOT:
        msgs.append({"role": "user", "content": flat(r["question"])})
        msgs.append({"role": "assistant", "content": flat(r["answer"])})
    msgs.append({"role": "user", "content": flat(source)})
    return msgs


print("Few-shot examples ({}), all from OUTSIDE the sample:".format(len(FEWSHOT)))
for r in FEWSHOT:
    print("  {} | {}".format(r["id"], flat(r["question"])[:62]))

print("\n" + "=" * 66)
print(build_completion(sample[0]["question"]))
print("=" * 66)
print("[reference: {}]".format(flat(sample[0]["answer"])))

Few-shot examples (3), all from OUTSIDE the sample:
  hinge_001881 | We're going to go back to the moon ... 50 years later?
  hinge_000262 | and we had thought that men and jinn would never speak against
  hinge_001687 | Whether the label widget should fill all available horizontal 

Rewrite English sentences as Hinglish - Hindi-English code-mixed text written in the Roman alphabet.

Translate some words and phrases into Hindi but write them in Roman letters, not Devanagari. Keep English words that Hindi speakers normally keep in English - technical terms, proper nouns, and common loanwords. The result should read the way an Indian speaker would naturally mix the two languages.

Output only the Hinglish sentence - no explanation, no Devanagari, no repetition of the English.

English: We're going to go back to the moon ... 50 years later?
Hinglish: 50 saal ke baad we’re going to go back to the moon.

English: and we had thought that men and jinn would never speak against God a lie.
Hing

### Cell 6 - Generate and score

Greedy decoding (`do_sample=False`) for reproducibility, with `max_new_tokens`
scaled to the source length.

`clean_output` strips the labels models add (`Hinglish:`, `Here is the...`) and
cuts anything from a following `English:` marker, which base models emit as
they continue the few-shot pattern. Left in, that boilerplate would depress
chrF++ for a formatting reason and be misread as a bad generation.

It does **not** strip Devanagari. A model that answers in Devanagari has failed
the task - HinGE's targets are romanised - so that should score badly rather
than be silently repaired. Cell 11 reports how often it happens, since a high
rate means the prompt needs work, not that the rows are hard.

In [13]:
_LABELS = [
    re.compile(r"^\s*(here(?:\s+is|'s)?\s+the\s+)?hinglish\s*(version|sentence)?\s*[:\-]\s*", re.I),
    re.compile(r"^\s*(sure|certainly)[,!]?\s*", re.I),
]
_DEV = re.compile(r"[\u0900-\u097F]")


def clean_output(text):
    t = (text or "").strip()
    t = re.split(r"\n\s*English\s*:", t)[0]     # base model ran on
    for line in t.split("\n"):
        line = line.strip()
        if not line:
            continue
        for pat in _LABELS:
            line = pat.sub("", line)
        line = line.strip().strip('"')
        if line:
            return line
    return ""


def prompt_style(tokenizer):
    # base checkpoints have no chat template at all
    return "chat" if getattr(tokenizer, "chat_template", None) else "completion"


@torch.no_grad()
def generate_hinglish(model, tokenizer, source):
    if prompt_style(tokenizer) == "completion":
        text = build_completion(source)
    else:
        msgs = build_chat_messages(source)
        try:
            text = tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True)
        except Exception:
            # some templates (Gemma) reject a system role - fold it into the
            # first user turn rather than dropping the instructions
            merged = [dict(m) for m in msgs[1:]]
            merged[0]["content"] = msgs[0]["content"] + "\n\n" + merged[0]["content"]
            text = tokenizer.apply_chat_template(
                merged, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    n_in   = inputs["input_ids"].shape[1]
    budget = min(3 * len(str(source).split()) + 64, MAX_NEW_TOKENS)

    out = model.generate(**inputs,
                         max_new_tokens=budget,
                         do_sample=False,
                         pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    return clean_output(tokenizer.decode(out[0][n_in:], skip_special_tokens=True))


def has_devanagari(text):
    return bool(_DEV.search(text or ""))


def clear_hf_cache():
    shutil.rmtree("/root/.cache/huggingface/hub/", ignore_errors=True)
    gc.collect()
    torch.cuda.empty_cache()


for raw, want in [
    ("Hinglish: yeh ek test hai", "yeh ek test hai"),
    ("Here is the Hinglish version: theek hai", "theek hai"),
    ("Sure, yeh sentence hai\nEnglish: next one", "yeh sentence hai"),
    ('"quoted output"', "quoted output"),
    ("", ""),
]:
    got = clean_output(raw)
    print("  clean {!r:<46} -> {!r}".format(raw, got))
    assert got == want, (raw, got, want)
print("\nGeneration and scoring functions defined")

  clean 'Hinglish: yeh ek test hai'                    -> 'yeh ek test hai'
  clean 'Here is the Hinglish version: theek hai'      -> 'theek hai'
  clean 'Sure, yeh sentence hai\nEnglish: next one'    -> 'yeh sentence hai'
  clean '"quoted output"'                              -> 'quoted output'
  clean ''                                             -> ''

Generation and scoring functions defined


### Cell 7 - Load-or-cache, and the batched runner

`load_model` implements download-once: if `models/<name>_4bit` exists in Drive
it is loaded directly (already 4-bit, so a fresh `BitsAndBytesConfig` would
conflict and is omitted); otherwise the repo is downloaded, quantised, and
saved to Drive. `trust_remote_code` stays off - repo-shipped modelling code is
often written against an older transformers API.

`run_model` stores the **raw chrF++ score and the generated text**, never a
pass/fail, which is what makes re-thresholding free. Each finished batch is
appended to `judge_progress/<model>.jsonl` before the next begins, so a
disconnect costs at most `BATCH_SIZE` rows.

In [14]:
def cache_path(spec):
    return os.path.join(CACHE_DIR, spec["name"] + "_4bit")


def load_model(spec):
    cached     = cache_path(spec)
    from_drive = DRIVE_OK and os.path.isfile(os.path.join(cached, "config.json"))
    source     = cached if from_drive else spec["repo"]

    kwargs = {"device_map": "auto", "trust_remote_code": False}
    if from_drive:
        how = "Drive cache (already 4-bit)"
    else:
        kwargs["quantization_config"] = bnb_config
        how = "HuggingFace download -> 4-bit"
    if spec.get("attn"):
        kwargs["attn_implementation"] = spec["attn"]
        how += ", attn=" + spec["attn"]

    print("  loading {} [{}]".format(source, how))
    tokenizer = AutoTokenizer.from_pretrained(source)
    model = AutoModelForCausalLM.from_pretrained(source, **kwargs).eval()

    if not from_drive and DRIVE_OK:
        print("  saving 4-bit copy to {} (one time)...".format(cached))
        os.makedirs(cached, exist_ok=True)
        model.save_pretrained(cached)
        tokenizer.save_pretrained(cached)
        print("  saved - future runs skip the download")

    print("  ready | VRAM: {:.2f}GB | prompt style: {}".format(
        torch.cuda.memory_allocated() / 1e9, prompt_style(tokenizer)))
    return model, tokenizer


def run_model(spec, rows):
    prog_file = os.path.join(PROG_DIR, spec["name"] + ".jsonl")

    done = {}
    if os.path.exists(prog_file):
        with open(prog_file, encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                done[item["id"]] = item
        print("  resuming - {}/{} already scored".format(len(done), len(rows)))

    remaining = [r for r in rows if r["id"] not in done]
    if not remaining:
        print("  {} already complete - skipping load".format(spec["name"]))
        return done

    model, tokenizer = load_model(spec)
    total_batches = (len(remaining) + BATCH_SIZE - 1) // BATCH_SIZE

    for batch_start in range(0, len(remaining), BATCH_SIZE):
        batch     = remaining[batch_start : batch_start + BATCH_SIZE]
        batch_num = batch_start // BATCH_SIZE + 1

        batch_results = []
        for row in batch:
            hyp = generate_hinglish(model, tokenizer, row["question"])
            batch_results.append({
                "id":      row["id"],
                "chrf":    chrf_pp(hyp, REFS[row["question"]]),
                "n_words": len(hyp.split()),
                "devan":   int(has_devanagari(hyp)),
                "output":  hyp,
            })

        with open(prog_file, "a", encoding="utf-8") as f:
            for item in batch_results:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

        done.update({item["id"]: item for item in batch_results})
        mean_chrf = sum(v["chrf"] for v in done.values()) / len(done)
        print("  batch {}/{} saved - {}/{} rows | mean chrF++ {:.1%}".format(
            batch_num, total_batches, len(done), len(rows), mean_chrf))

    del model, tokenizer
    clear_hf_cache()
    print("  {} complete".format(spec["name"]))
    return done

print("Runner defined")

Runner defined


### Cell 8 - Run all three models

One model at a time - loaded, scored, unloaded - so peak VRAM stays near 6 GB
rather than the ~17 GB all three would need together.

The long cell: roughly **8-12 min per model**, plus downloads on the first run.
Safe to re-run - anything already scored is skipped.

In [15]:
preds = {}
for spec in MODELS:
    print("\n=== {} ===".format(spec["name"]))
    preds[spec["name"]] = run_model(spec, sample)

print("\nAll models done")


=== mistral ===
  loading /drive/MyDrive/models/mistral_4bit [Drive cache (already 4-bit)]


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  ready | VRAM: 4.14GB | prompt style: chat
  batch 1/8 saved - 25/200 rows | mean chrF++ 27.0%
  batch 2/8 saved - 50/200 rows | mean chrF++ 25.8%
  batch 3/8 saved - 75/200 rows | mean chrF++ 26.5%
  batch 4/8 saved - 100/200 rows | mean chrF++ 27.0%
  batch 5/8 saved - 125/200 rows | mean chrF++ 26.4%
  batch 6/8 saved - 150/200 rows | mean chrF++ 27.2%
  batch 7/8 saved - 175/200 rows | mean chrF++ 26.9%
  batch 8/8 saved - 200/200 rows | mean chrF++ 27.1%
  mistral complete

=== llama ===
  loading /drive/MyDrive/models/llama_4bit [Drive cache (already 4-bit)]


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  ready | VRAM: 5.71GB | prompt style: chat


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  batch 1/8 saved - 25/200 rows | mean chrF++ 30.2%
  batch 2/8 saved - 50/200 rows | mean chrF++ 29.9%
  batch 3/8 saved - 75/200 rows | mean chrF++ 30.1%
  batch 4/8 saved - 100/200 rows | mean chrF++ 30.0%
  batch 5/8 saved - 125/200 rows | mean chrF++ 30.2%
  batch 6/8 saved - 150/200 rows | mean chrF++ 30.3%
  batch 7/8 saved - 175/200 rows | mean chrF++ 30.0%
  batch 8/8 saved - 200/200 rows | mean chrF++ 30.4%
  llama complete

=== gemma ===
  loading /drive/MyDrive/models/gemma_4bit [Drive cache (already 4-bit), attn=eager]


Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

  ready | VRAM: 6.14GB | prompt style: chat
  batch 1/8 saved - 25/200 rows | mean chrF++ 30.4%
  batch 2/8 saved - 50/200 rows | mean chrF++ 29.7%
  batch 3/8 saved - 75/200 rows | mean chrF++ 30.7%
  batch 4/8 saved - 100/200 rows | mean chrF++ 31.1%
  batch 5/8 saved - 125/200 rows | mean chrF++ 30.6%
  batch 6/8 saved - 150/200 rows | mean chrF++ 30.9%
  batch 7/8 saved - 175/200 rows | mean chrF++ 30.6%
  batch 8/8 saved - 200/200 rows | mean chrF++ 31.1%
  gemma complete

All models done


### Cell 9 - Find the threshold

The pass mark is derived from the data, not guessed. The cell prints the score
distribution per model and pooled, places the model-free floors alongside, and
shows what every candidate threshold does to the Easy / Medium / Hard split.

**On this dataset the floor is the binding constraint.** With a ~56% floor,
`auto_median` can easily land just above it and pass models that barely
translated anything, so `floor_margin` is the default: the pass mark is the
floor plus `FLOOR_MARGIN`, which asks "did this model beat *doing nothing* by a
real margin?"

Warnings fire if the chosen threshold falls at or below the floor, sits so high
that only near-exact matches pass, or empties a difficulty band.

Nothing here re-runs a model, so you can change `THRESHOLD_MODE` in Cell 3 and
re-run just this cell and the next.

In [16]:
pooled = sorted(preds[s["name"]][r["id"]]["chrf"] for r in sample for s in MODELS)

def pct(p):
    return pooled[min(len(pooled) - 1, int(p * len(pooled)))]

print("chrF++ distribution per model:")
print("  {:<10} {:>7} {:>7} {:>7} {:>7}".format("model", "p25", "median", "p75", "mean"))
for s in MODELS:
    v = sorted(x["chrf"] for x in preds[s["name"]].values())
    print("  {:<10} {:>6.1%} {:>7.1%} {:>7.1%} {:>7.1%}".format(
        s["name"], v[len(v) // 4], v[len(v) // 2], v[3 * len(v) // 4],
        sum(v) / len(v)))
print("  {:<10} {:>6.1%} {:>7.1%} {:>7.1%} {:>7.1%}".format(
    "POOLED", pct(.25), pct(.50), pct(.75), sum(pooled) / len(pooled)))

print("\nmodel-free floors (from Cell 4):")
print("  copy the input : {:.1%}".format(COPY_FLOOR))
print("  unrelated text : {:.1%}".format(RAND_FLOOR))


def difficulty_at(th):
    out = Counter()
    for r in sample:
        votes = sum(preds[s["name"]][r["id"]]["chrf"] >= th for s in MODELS)
        out["Easy" if votes == 3 else ("Medium" if votes == 2 else "Hard")] += 1
    return out


if THRESHOLD_MODE == "floor_margin":
    THRESHOLD = COPY_FLOOR + FLOOR_MARGIN
    why = "copy-the-input floor + {:.2f} margin".format(FLOOR_MARGIN)
elif THRESHOLD_MODE == "auto_median":
    THRESHOLD = pct(.50)
    why = "median of all pooled model scores"
elif THRESHOLD_MODE == "fixed":
    THRESHOLD = FIXED_THRESHOLD
    why = "FIXED_THRESHOLD from Cell 3"
else:
    raise ValueError("unknown THRESHOLD_MODE: " + str(THRESHOLD_MODE))

print("\nsensitivity - what each threshold would produce:")
print("  {:>9}  {:>6} {:>7} {:>6}".format("threshold", "Easy", "Medium", "Hard"))
cands = sorted(set(round(x, 3) for x in
                   [.50, .55, .60, .65, .70, .75, .80,
                    round(COPY_FLOOR, 3), round(THRESHOLD, 3)]))
for th in cands:
    d = difficulty_at(th)
    tag = ""
    if abs(th - round(THRESHOLD, 3)) < 1e-9:
        tag += "  <- CHOSEN"
    if abs(th - round(COPY_FLOOR, 3)) < 1e-9:
        tag += "  (copy-input floor)"
    print("  {:>9.3f}  {:>6} {:>7} {:>6}{}".format(
        th, d.get("Easy", 0), d.get("Medium", 0), d.get("Hard", 0), tag))

print("\nTHRESHOLD = {:.3f}  ({})".format(THRESHOLD, why))

if THRESHOLD <= COPY_FLOOR:
    print("  WARNING: at or below the copy-the-input floor - a model could pass")
    print("  without translating anything. Use THRESHOLD_MODE='floor_margin'.")
else:
    print("  sits {:.1f} points above the copy-the-input floor - OK".format(
        100 * (THRESHOLD - COPY_FLOOR)))

if THRESHOLD >= 0.95:
    print("  WARNING: the threshold sits at the very top of the range and")
    print("  demands a near-exact match. Use THRESHOLD_MODE='fixed'.")

_bands = difficulty_at(THRESHOLD)
if min(_bands.get(k, 0) for k in ("Easy", "Medium", "Hard")) == 0:
    print("  WARNING: one difficulty band is empty at this threshold - the")
    print("  split carries little information. Pick another value.")

chrF++ distribution per model:
  model          p25  median     p75    mean
  mistral     17.7%   24.7%   35.7%   27.1%
  llama       19.9%   28.2%   37.6%   30.4%
  gemma       20.8%   29.7%   38.6%   31.1%
  POOLED      19.6%   27.4%   37.3%   29.5%

model-free floors (from Cell 4):
  copy the input : 48.4%
  unrelated text : 11.3%

sensitivity - what each threshold would produce:
  threshold    Easy  Medium   Hard
      0.484       4       7    189  (copy-input floor)
      0.500       3       7    190
      0.550       1       5    194
      0.584       1       3    196  <- CHOSEN
      0.600       1       1    198
      0.650       0       1    199
      0.700       0       1    199
      0.750       0       1    199
      0.800       0       1    199

THRESHOLD = 0.584  (copy-the-input floor + 0.10 margin)
  sits 10.0 points above the copy-the-input floor - OK


### Cell 10 - Apply the threshold and write the schema

Each model votes 1 where its chrF++ clears `THRESHOLD`; the votes sum into
Easy / Medium / Hard as in every other split.

Output rows are rebuilt key-by-key from `SCHEMA_KEYS`, so the file carries
exactly the 14 IndicSample fields in schema order. `difficulty` is the only
value that changes unless you set `SET_EVAL_METRIC`. Generations and raw scores
go to the audit file.

In [17]:
def get_difficulty(votes):
    score = sum(votes)
    if score == 3:
        return "Easy"
    elif score == 2:
        return "Medium"
    else:
        return "Hard"


final_results = []
audit = []

for row in sample:
    scores = [preds[s["name"]][row["id"]]["chrf"] for s in MODELS]
    votes  = [int(x >= THRESHOLD) for x in scores]
    difficulty = get_difficulty(votes)

    enriched = {**row, "difficulty": difficulty}
    if SET_EVAL_METRIC:
        enriched["eval_metric"] = SET_EVAL_METRIC
    final_results.append({k: enriched.get(k) for k in SCHEMA_KEYS})

    audit.append({
        "id":          row["id"],
        "difficulty":  difficulty,
        "votes":       votes,
        "chrf":        [round(x, 4) for x in scores],
        "threshold":   round(THRESHOLD, 4),
        "english":     " ".join(row["question"].split()),
        "references":  [" ".join(x.split()) for x in REFS[row["question"]]],
        "generations": {s["name"]: preds[s["name"]][row["id"]]["output"]
                        for s in MODELS},
    })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for item in final_results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(AUDIT_FILE, "w", encoding="utf-8") as f:
    for item in audit:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Saved -> {} ({} rows)".format(OUTPUT_FILE, len(final_results)))
print("Audit -> {}".format(AUDIT_FILE))
print("threshold used: {:.3f}".format(THRESHOLD))

Saved -> hinge_difficulty.jsonl (200 rows)
Audit -> hinge_audit.jsonl
threshold used: 0.584


### Cell 11 - Verify and report

Checks before trusting the file:

1. **Schema** - all 14 keys in order, no nulls in `difficulty`.
2. **Difficulty distribution.**
3. **Per-model mean chrF++ against the copy-input floor.** On HinGE this is the
   check that matters: a model *at* the floor has produced English, not
   Hinglish, and its votes are meaningless.
4. **Devanagari rate.** A model writing Devanagari has failed the task -
   HinGE's targets are romanised. A high rate is a prompting problem, not a
   difficulty signal.
5. **Empty-output rate**, same reasoning.

The sample rows print the English, every reference, and all three generations,
which is the quickest way to sanity-check that Hard rows are genuinely hard.

In [18]:
bad_keys = [r["id"] for r in final_results if list(r.keys()) != SCHEMA_KEYS]
missing  = [r["id"] for r in final_results if r["difficulty"] is None]
print("Schema check : {} rows | wrong keys: {} | null difficulty: {}".format(
    len(final_results), len(bad_keys), len(missing)))

dist  = Counter(r["difficulty"] for r in final_results)
total = len(final_results)
print("\nDifficulty distribution (threshold {:.3f}):".format(THRESHOLD))
for level in ["Easy", "Medium", "Hard"]:
    n = dist.get(level, 0)
    print("  {:<7}: {:4d}  ({:.1f}%)".format(level, n, n / total * 100))

print("\nMean chrF++:")
for s in MODELS:
    v = [x["chrf"] for x in preds[s["name"]].values()]
    m = sum(v) / len(v)
    flag = "  <- at/below the do-nothing floor" if m <= COPY_FLOOR else ""
    print("  {:<10} {:.1%}{}".format(s["name"], m, flag))
print("  {:<10} {:.1%}  <- copy-the-input floor".format("baseline", COPY_FLOOR))

print("\nOutput health:")
for s in MODELS:
    v = list(preds[s["name"]].values())
    empty = sum(1 for x in v if x["n_words"] == 0)
    dev   = sum(x["devan"] for x in v)
    flags = []
    if dev / total > 0.10:
        flags.append("Devanagari - prompt problem")
    if empty / total > 0.05:
        flags.append("empty outputs")
    print("  {:<10} empty {:>3}/{} | devanagari {:>3}/{} | median words {}{}".format(
        s["name"], empty, total, dev, total,
        sorted(x["n_words"] for x in v)[len(v) // 2],
        "  <- " + ", ".join(flags) if flags else ""))

print("\n--- 2 sample rows ---")
for a in audit[:2]:
    print("\n  {} [{}] chrf={}".format(a["id"], a["difficulty"], a["chrf"]))
    print("    english   :", a["english"][:88])
    for ref in a["references"]:
        print("    reference :", ref[:88])
    for k, v in a["generations"].items():
        print("    {:<9} :".format(k), v[:88])

Schema check : 200 rows | wrong keys: 0 | null difficulty: 0

Difficulty distribution (threshold 0.584):
  Easy   :    1  (0.5%)
  Medium :    3  (1.5%)
  Hard   :  196  (98.0%)

Mean chrF++:
  mistral    27.1%  <- at/below the do-nothing floor
  llama      30.4%  <- at/below the do-nothing floor
  gemma      31.1%  <- at/below the do-nothing floor
  baseline   48.4%  <- copy-the-input floor

Output health:
  mistral    empty   0/200 | devanagari   0/200 | median words 16
  llama      empty   0/200 | devanagari   0/200 | median words 15
  gemma      empty   0/200 | devanagari   0/200 | median words 13

--- 2 sample rows ---

  hinge_001314 [Hard] chrf=[0.1193, 0.4437, 0.6191]
    english   : And we 're going to expand it into two simpler expressions
    reference : And we 're going to expand do saral bhav me.
    mistral   : Hum bhi wapas bhi chale jayenge bimaroon par... 50 saal baad.
    llama     : And we’re going to expand it into do simpler expressions.
    gemma     : And we 're 